In [1]:
import pandas as pd
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
train_df = pd.read_parquet("../data/processed/casp9_features.parquet")
val_df = pd.read_parquet("../data/processed/validation_features.parquet")
test_df = pd.read_parquet("../data/processed/testing_features.parquet")


print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

Train: (3337092, 51)
Val: (47602, 51)
Test: (24466, 51)


Distance Target Generator 

In [3]:
def add_distance_targets(df):

    df = df.sort_values(["ProteinID","ResidueIndex"]).copy()

    coords = df[["X","Y","Z"]].values

    # shift coordinates for neighbors
    coords_i1 = np.roll(coords, -1, axis=0)
    coords_i2 = np.roll(coords, -2, axis=0)
    coords_i4 = np.roll(coords, -4, axis=0)
    coords_i8 = np.roll(coords, -8, axis=0)

    df["dist_i1"] = np.linalg.norm(coords - coords_i1, axis=1)
    df["dist_i2"] = np.linalg.norm(coords - coords_i2, axis=1)
    df["dist_i4"] = np.linalg.norm(coords - coords_i4, axis=1)
    df["dist_i8"] = np.linalg.norm(coords - coords_i8, axis=1)

    return df

In [4]:
train_df = add_distance_targets(train_df)
val_df = add_distance_targets(val_df)
test_df = add_distance_targets(test_df)

### Remove invalid edges 



Residues near sequence ends cannot have neighbors.

In [5]:
target_cols = ["dist_i1","dist_i2","dist_i4","dist_i8"]

train_df = train_df.dropna(subset=target_cols)
val_df = val_df.dropna(subset=target_cols)
test_df = test_df.dropna(subset=target_cols)

print(train_df.shape)

(3337092, 55)


In [7]:
train_df.to_parquet("../data/Multi_distance_edits/train_dist.parquet")
val_df.to_parquet("../data/Multi_distance_edits/val_dist.parquet")
test_df.to_parquet("../data/Multi_distance_edits/test_dist.parquet")

In [8]:
feature_cols = [c for c in train_df.columns if c.startswith("Evo") or c.startswith("AA_")]

print(len(feature_cols))

41


In [9]:
WINDOW = 5
WINDOW_SIZE = 2 * WINDOW + 1

In [10]:
def build_windows_fast(df):

    X_list = []
    y_list = []

    for pid, group in df.groupby("ProteinID"):

        group = group.sort_values("ResidueIndex")

        features = group[feature_cols].values
        targets = group[["dist_i1","dist_i2","dist_i4","dist_i8"]].values

        padded = np.pad(features, ((WINDOW,WINDOW),(0,0)), mode="constant")

        windows = sliding_window_view(
            padded,
            (WINDOW_SIZE, features.shape[1])
        )

        windows = windows.reshape(len(group), -1)

        X_list.append(windows)
        y_list.append(targets)

    X = np.vstack(X_list)
    y = np.vstack(y_list)

    return X, y

Generating the sliding windows

In [11]:
X_train_w, y_train_w = build_windows_fast(train_df)

print(X_train_w.shape, y_train_w.shape)

(3337092, 451) (3337092, 4)


In [12]:
X_val_w, y_val_w = build_windows_fast(val_df)

In [13]:
X_test_w, y_test_w = build_windows_fast(test_df)

Saving the files 

In [18]:
np.save("..\data\Saved_Windows_features\X_train.npy", X_train_w)
np.save("..\data\Saved_Windows_features\y_train.npy", y_train_w)

np.save("..\data\Saved_Windows_features\X_val.npy", X_val_w)
np.save("..\data\Saved_Windows_features\y_val.npy", y_val_w)

np.save("..\data\Saved_Windows_features\X_test.npy", X_test_w)
np.save("..\data\Saved_Windows_features\y_test.npy", y_test_w)

## Training the Model

In [14]:
xgb_model = XGBRegressor(

    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    tree_method="hist",
    device="cuda",

    max_bin=256,

    random_state=42
)

In [15]:
xgb_model.fit(
    X_train_w,
    y_train_w,
    eval_set=[(X_val_w, y_val_w)],
    verbose=True
)

[0]	validation_0-rmse:1095.92067
[1]	validation_0-rmse:1075.43735
[2]	validation_0-rmse:1054.86733
[3]	validation_0-rmse:1036.80133
[4]	validation_0-rmse:1019.65402
[5]	validation_0-rmse:1003.44408
[6]	validation_0-rmse:988.69873
[7]	validation_0-rmse:974.48251
[8]	validation_0-rmse:961.52298
[9]	validation_0-rmse:950.60953
[10]	validation_0-rmse:940.41145
[11]	validation_0-rmse:930.57385
[12]	validation_0-rmse:921.77289
[13]	validation_0-rmse:912.61904
[14]	validation_0-rmse:904.58145
[15]	validation_0-rmse:897.50777
[16]	validation_0-rmse:891.78137
[17]	validation_0-rmse:885.47993
[18]	validation_0-rmse:879.25618
[19]	validation_0-rmse:873.95524
[20]	validation_0-rmse:868.73780
[21]	validation_0-rmse:864.53748
[22]	validation_0-rmse:860.07569
[23]	validation_0-rmse:855.96648
[24]	validation_0-rmse:852.07013
[25]	validation_0-rmse:848.44707
[26]	validation_0-rmse:846.08103
[27]	validation_0-rmse:843.18436
[28]	validation_0-rmse:840.88409
[29]	validation_0-rmse:838.48064
[30]	validatio

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,'cuda'
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [16]:
preds = xgb_model.predict(X_test_w)

d:\ProteinPrediction\MainFiles\GPU_VENV\lib\site-packages\xgboost\core.py:751: UserWarning: [15:58:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [17]:
rmse = np.sqrt(mean_squared_error(y_test_w, preds))
mae = mean_absolute_error(y_test_w, preds)
r2 = r2_score(y_test_w, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 629.3886418581129
MAE: 175.75497436523438
R2: 0.6281043291091919


In [18]:
import joblib

joblib.dump(xgb_model, "xgboost(SW)_multiDistance_model.pkl")

['xgboost(SW)_multiDistance_model.pkl']